# 🤖 LangChain + Bigdata MCP Integration

This notebook demonstrates how your AI agents can interact with **Bigdata.com via MCP (Model Context Protocol)**—a standardized way to connect AI applications to data sources and tools with automatic discovery.

## What This Demonstrates

**Bigdata.com MCP Integration:**
- **Automatic Tool Discovery** → MCP exposes all available tools dynamically; when Bigdata.com adds new capabilities (tearsheets, calendars, screeners), your agent gets them automatically
- **Company Lookup** → Resolve tickers to entity IDs via Knowledge Graph
- **Search** → Query news, filings, transcripts with filters
- **Tearsheets** → Company and country financial profiles
- **Events Calendar** → Earnings dates, conference calls

**Internal Data Integration:**
- Connect to your portfolio databases (positions, transactions, P&L)
- Semantic search over internal research documents via vector stores
- Combine MCP-discovered tools with your internal tools seamlessly

**Framework Flexibility:**
> This demo uses **LangChain** with **langchain-mcp-adapters** and **LangSmith** for observability. The MCP protocol is framework-agnostic—**CrewAI**, **AutoGen**, **Google A2A**, and other frameworks can connect to MCP servers using their respective adapters. The key benefit: one integration, automatic access to all current and future Bigdata.com tools.

---

## What is MCP (Model Context Protocol)?

MCP is an open protocol that standardizes how AI applications connect to data sources and tools. Think of it as "USB for AI" - a universal connector.

**Key Benefits:**
- **Automatic Tool Discovery**: MCP servers expose tools dynamically - no manual updates needed when new tools are added
- **Standardized Interface**: One protocol works across all MCP-compatible tools
- **Stateful Connections**: Efficient communication with long-lived sessions

## Architecture

```
┌─────────────────────────────────────────────────────────┐
│                  LangChain Agent                        │
│  (ReAct Pattern - Reasoning + Acting)                   │
└────────────┬──────────────┬──────────────┬──────────────┘
             │              │              │
             ▼              ▼              ▼
      ┌────────────┐ ┌───────────┐ ┌────────────────┐
      │  Local DB  │ │  FAISS    │ │ Bigdata MCP    │
      │  (SQLite)  │ │  Vector   │ │ Server         │
      │            │ │  Store    │ │                │
      └────────────┘ └───────────┘ └────────────────┘
```

---

## 1️⃣ Install Dependencies

Latest versions of LangChain and LangChain MCP Adapters:

In [1]:
%pip install langchain langchain-openai langchain-community faiss-cpu langchain-mcp-adapters python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


## 2️⃣ Import Libraries

**LangChain**: ReAct agent and tool-calling

**LangChain MCP Adapters**: Bridge between LangChain and MCP servers

**langgraph_core** (reusable): Environment, `create_financial_database`, `create_vector_store`, local tools (`get_database_tools`, `get_vectorstore_tools`), and display helpers (`display_query`, `display_response`, `display_tools_used`, `display_citations`)

In [2]:
import os
import json
import sqlite3
import random
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from dotenv import load_dotenv

# Display utilities for Jupyter
from IPython.display import display, Markdown, HTML
import html as html_lib

# LangChain
from langchain.tools import tool
from langchain.agents import create_agent as langchain_create_agent
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# MCP integration
from langchain_mcp_adapters.client import MultiServerMCPClient

# Reusable core (langgraph_core): environment, data sources, display
import sys
sys.path.append(".")
from langgraph_core import (
    setup_environment,
    create_financial_database,
    create_vector_store,
    get_database_tools,
    get_vectorstore_tools,
    display_query,
    display_response,
    display_tools_used,
    display_citations,
)
load_dotenv()
print("✅ Libraries imported successfully")

/Users/bakulkumarkakadiya/dev/github/bigdata-cookbook/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Libraries imported successfully


In [3]:
# Environment, database, vector store, and display helpers are provided by langgraph_core
# (see langgraph_core.py). No local definitions needed for reusability.
print("✅ Using langgraph_core for environment, data sources, and display helpers")


✅ Using langgraph_core for environment, data sources, and display helpers


## 3️⃣ Setup Environment & Local Data Sources

Initialize:
- LangSmith tracing for observability
- Local SQLite database with sample portfolio data
- FAISS vector store with research documents

In [4]:
# Setup environment (loads API keys, enables LangSmith tracing)
config = setup_environment(
    langsmith_project="langgraph-bigdata-mcp-demo",
    enable_tracing=True
)

# Create local database with sample financial data
create_financial_database()

# Create vector store with research documents
create_vector_store()

print("\n✅ Local data sources ready")

✅ LangSmith tracing enabled → Project: langgraph-bigdata-mcp-demo
✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...
✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

✅ Local data sources ready


## 4️⃣ Load Local Tools

Load tools that interact with local data sources:

In [5]:
# Get local database tools
local_db_tools = get_database_tools()
print(f"✅ Loaded {len(local_db_tools)} database tools:")
for t in local_db_tools:
    print(f"   - {t.name}: {t.description[:80]}...")

# Get local vector store tools
local_vector_tools = get_vectorstore_tools()
print(f"\n✅ Loaded {len(local_vector_tools)} vector store tools:")
for t in local_vector_tools:
    print(f"   - {t.name}: {t.description[:80]}...")

✅ Loaded 2 database tools:
   - internal_query_database: Execute SQL query against the internal financial transactions database.

Availab...
   - internal_portfolio_summary: Get a summary of a specific portfolio from internal database including holdings ...

✅ Loaded 1 vector store tools:
   - internal_search_research: Search internal research documents using semantic similarity.

This searches thr...


## 5️⃣ Connect to Bigdata MCP Server

**MCP Configuration:**
- **URL**: `https://mcp.bigdata.com/`
- **Transport**: HTTP (streamable)
- **Authentication**: `x-api-key` header

The MCP client automatically discovers all available tools from the server.

In [6]:
# API keys
BIGDATA_API_KEY = os.getenv("BIGDATA_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not BIGDATA_API_KEY:
    raise ValueError("BIGDATA_API_KEY not found. Set via: export BIGDATA_API_KEY='your-key'")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Set via: export OPENAI_API_KEY='your-key'")

print(f"✅ Bigdata API Key: {BIGDATA_API_KEY[:10]}...")
print(f"✅ OpenAI API Key: {OPENAI_API_KEY[:10]}...")

✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...


### Configure MCP Client

**Important**: Set `ANYIO_BACKEND='asyncio'` to ensure async backend detection in Jupyter:

In [7]:
# Set async backend for anyio (required in Jupyter)
os.environ['ANYIO_BACKEND'] = 'asyncio'

# Configure MCP client
mcp_client = MultiServerMCPClient(
    {
        "bigdata": {
            "url": "https://mcp.bigdata.com/",
            "transport": "http",  # Streamable HTTP transport
            "headers": {
                "x-api-key": BIGDATA_API_KEY
            }
        }
    }
)

print("✅ MCP client configured")

✅ MCP client configured


### Load MCP Tools

The MCP client automatically discovers all tools exposed by the Bigdata MCP server:

**Note**: In Jupyter notebooks, we use `await` directly instead of `asyncio.run()` because Jupyter already runs an event loop in the background.

In [8]:
# Load tools from Bigdata MCP server
# Note: In Jupyter notebooks, there's already a running event loop, so we use 'await' directly
print("Connecting to Bigdata MCP...")
bigdata_mcp_tools = await mcp_client.get_tools()
print(f"✅ Loaded {len(bigdata_mcp_tools)} tools from Bigdata MCP:")
for tool in bigdata_mcp_tools:
    print(f"   - {tool.name}: {tool.description[:80]}...")

Connecting to Bigdata MCP...
✅ Loaded 5 tools from Bigdata MCP:
   - bigdata_country_tearsheet: Returns a comprehensive country economic tearsheet with economic calendar data.
...
   - bigdata_events_calendar: Returns a professionally formatted markdown calendar of corporate events includi...
   - find_companies: REQUIRED FIRST STEP: Run this tool whenever the user mentions a company for the ...
   - bigdata_search: Search engine for financial documents, earnings call transcripts, news articles,...
   - bigdata_company_tearsheet: Returns a comprehensive company tearsheet with financial data, market intelligen...


## 6️⃣ Combine All Tools

Merge tools from all sources:

In [9]:
# Combine all tools
all_tools = local_db_tools + local_vector_tools + bigdata_mcp_tools

print(f"\n✅ Total tools available: {len(all_tools)}")
print(f"   - Local DB tools: {len(local_db_tools)}")
print(f"   - Local vector store tools: {len(local_vector_tools)}")
print(f"   - Bigdata MCP tools: {len(bigdata_mcp_tools)}")


✅ Total tools available: 8
   - Local DB tools: 2
   - Local vector store tools: 1
   - Bigdata MCP tools: 5


## 7️⃣ Create LangChain Agent

**LangChain ReAct Agent:**
- **Reasoning**: Plans which tools to use based on user query
- **Acting**: Executes tool calls and processes results
- **Iteration**: Continues until query is fully answered

Uses `langchain.agents.create_agent` (the current, non-deprecated API) which creates an agent with:
- Agent executor (LLM with tool calling capabilities)
- Tool registry (all available tools)
- System prompt (guides agent behavior)
- Streaming support (for real-time responses)

In [10]:
# Define system prompt for the agent
SYSTEM_PROMPT = """You are an intelligent financial research assistant with access to multiple data sources:

**External Data (Bigdata.com MCP):**
- Tools dynamically loaded from Bigdata MCP server
- Tools include news, prices, tear sheet, search, company lookup, research agent, and other capital markets capabilities

**Internal Data (Company Systems):**
- `internal_query_database` - Execute SQL queries on portfolio/transaction database
- `internal_portfolio_summary` - Get portfolio holdings and performance summary
- `internal_search_research` - Search internal investment research documents

Guidelines:
- Use appropriate tools based on the query
- For portfolio questions, use internal database tools
- For market intelligence, use Bigdata MCP tools
- Combine multiple sources for comprehensive analysis
- Cite sources when using external data

Available portfolios: PF001 (US Large Cap Growth), PF002 (AI & Semiconductor Focus), PF003 (Diversified Tech Leaders)
"""

# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=OPENAI_API_KEY
)

# Create agent using langchain.agents.create_agent (non-deprecated)
agent = langchain_create_agent(llm, all_tools, system_prompt=SYSTEM_PROMPT)

print(f"✅ Agent created with {len(all_tools)} tools")
print(f"   Model: gpt-4o")
print(f"   System prompt configured")

✅ Agent created with 8 tools
   Model: gpt-4o
   System prompt configured


## 8️⃣ Run Example Queries

Let's test the agent with queries that utilize different data sources:

### Example 1 : Multinode

In [11]:
query = """
Analyze the AI & Semiconductor Focus portfolio (PF002):
1. What are our current holdings and their performance?
2. What risks does our internal research identify?
3. For each holding, get us negative news
"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)

# Display citations from Bigdata.com sources
display_citations(result)

### AI & Semiconductor Focus Portfolio (PF002) Analysis

#### Current Holdings and Performance
The AI & Semiconductor Focus portfolio (PF002) has the following holdings:

1. **NVIDIA Corporation (NVDA)**
   - Shares: 12,000
   - Average Cost: $450.00
   - Current Price: $875.50
   - Market Value: $10,506,000
   - Unrealized P&L: $5,106,000

2. **Broadcom Inc. (AVGO)**
   - Shares: 1,500
   - Average Cost: $850.00
   - Current Price: $1,425.00
   - Market Value: $2,137,500
   - Unrealized P&L: $862,500

3. **Palantir Technologies (PLTR)**
   - Shares: 25,000
   - Average Cost: $18.50
   - Current Price: $65.25
   - Market Value: $1,631,250
   - Unrealized P&L: $1,168,750

4. **Advanced Micro Devices (AMD)**
   - Shares: 8,000
   - Average Cost: $95.00
   - Current Price: $145.25
   - Market Value: $1,162,000
   - Unrealized P&L: $402,000

5. **Taiwan Semiconductor (TSM)**
   - Shares: 3,000
   - Average Cost: $110.00
   - Current Price: $185.75
   - Market Value: $557,250
   - Unrealized P&L: $227,250

**Total Market Value:** $15,994,000  
**Total Unrealized P&L:** $7,766,500

#### Identified Risks from Internal Research
1. **Valuation Risk (High):** High forward P/E ratios, especially in AI sectors, could lead to corrections if growth expectations are not met.
2. **Regulatory Risk (Medium-High):** Potential impacts from antitrust actions and digital market regulations, particularly affecting companies like Google and Apple.
3. **China Exposure (Medium):** Significant revenue exposure to China, with potential risks from export controls affecting companies like NVIDIA.
4. **AI Bubble Risk (Medium):** Concerns about the sustainability of AI infrastructure investments and their return on investment.

#### Negative News for Each Holding

1. **NVIDIA Corporation (NVDA)**
   - **Patent Infringement Lawsuit:** Health Discovery Corporation has filed a lawsuit against NVIDIA for patent infringement related to machine-learning methods [Benzinga - Jan 21, 2026](https://www.benzinga.com/node/50043634?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
   - **Regulatory Challenges in China:** NVIDIA is facing challenges in selling its AI technology in China due to regulatory issues [Insider Monkey - Jan 29, 2026](https://www.insidermonkey.com/blog/ceo-of-nvidia-corporation-nvda-in-shanghai-despite-regulatory-challenges-in-china-1683473/).

2. **Broadcom Inc. (AVGO)**
   - **Security Directive Impact:** Broadcom's stock has been affected by a Chinese directive to stop using certain cybersecurity software, impacting its market position [Benzinga - Jan 14, 2026](https://www.benzinga.com/node/49918896?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

3. **Palantir Technologies (PLTR)**
   - **Internal Controversy:** Palantir faces internal controversy over its work with U.S. Immigration and Customs Enforcement, affecting employee morale [Benzinga - Jan 28, 2026](https://www.benzinga.com/node/50203879?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

4. **Advanced Micro Devices (AMD)**
   - **Bearish Options Activity:** There is notable bearish sentiment in options trading for AMD, indicating potential concerns about its future performance [Benzinga - Jan 20, 2026](https://www.benzinga.com/node/50006193?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

5. **Taiwan Semiconductor (TSM)**
   - **Capacity Issues:** TSMC is facing capacity constraints, which could impact its ability to meet demand, creating opportunities for competitors like Intel [Benzinga - Jan 29, 2026](https://www.benzinga.com/node/50244148?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

### Sources
- Benzinga
- Edgar SEC Filings
- Insider Monkey
- MT Newswires
- Nasdaq

This analysis provides a comprehensive view of the current state and challenges facing the AI & Semiconductor Focus portfolio.

### Example 2: Local Database Query

Query internal portfolio holdings:

In [12]:
# Example 1: Query internal database for top holdings
query = "What are our top 5 holdings by market value across all portfolios?"

# Display query
display_query(query)

# Run agent (async invocation in Jupyter)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)

# Display citations from Bigdata.com sources
display_citations(result)

The top 5 holdings by market value across all portfolios are:

1. **NVIDIA Corporation (NVDA)** - $17,510,000
2. **Microsoft Corporation (MSFT)** - $9,556,500
3. **Apple Inc. (AAPL)** - $7,410,000
4. **Salesforce Inc. (CRM)** - $3,255,000
5. **Meta Platforms Inc. (META)** - $2,632,500

### Example 3: Local Vector Store Query

Search internal research documents:

In [13]:
# Example 2: Search internal vector store
query = "What does our internal research say about NVIDIA's competitive moat?"

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)

# Display citations from Bigdata.com sources
display_citations(result)

Our internal research highlights several aspects of NVIDIA's competitive moat:

1. **Software Ecosystem**: NVIDIA's CUDA ecosystem is a significant part of its competitive moat, with over 4 million developers. This creates substantial switching costs for developers and enterprises, making it difficult for competitors to lure them away. The widespread adoption of CUDA enhances NVIDIA's position in the AI and data center markets.

2. **Product Leadership**: NVIDIA's next-generation Blackwell Architecture, with the upcoming B100/B200 GPUs, promises a 2.5x performance improvement. This technological edge is crucial in maintaining its leadership in the GPU market, particularly for AI training and inference workloads.

3. **Market Opportunity**: NVIDIA is well-positioned to capitalize on the AI inference market, which is projected to have a $150 billion total addressable market (TAM) by 2027. This opportunity further strengthens its competitive position as enterprises increasingly deploy AI at scale.

4. **Data Center Revenue Growth**: NVIDIA's data center revenue has seen significant growth, driven by the demand for its H100/H200 GPUs. This growth underscores its strong foothold in the data center segment, a critical area for future expansion.

These factors collectively contribute to NVIDIA's strong competitive moat, supported by its technological advancements, robust software ecosystem, and strategic market positioning. However, potential risks include export restrictions to China, competition from AMD, and supply constraints.

### Example 4: Bigdata MCP Tool Query

Use external market intelligence:

In [14]:
# Example 3: Use Bigdata MCP tools for external data
query = "Find the latest news about NVIDIA's earnings and revenue growth using Bigdata tools."

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
#display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

In [18]:
# Display response
display_response(result)

# Display citations from Bigdata.com sources
display_citations(result)

Here's a summary of recent developments for CoreWeave over the last 30 days, categorized by type of event:

### Financial Results
- **Date:** January 29, 2026
- **Facts:** CoreWeave's stock price has seen significant volatility, with a 30.90% increase over the past month. The company reported a Q3 2025 EPS of -$0.08, beating estimates of -$0.40, and revenue of $1.36 billion, surpassing the expected $1.29 billion.
- **Investment Implications:** Bullish. The positive earnings surprise and revenue beat suggest stronger-than-expected financial performance, potentially boosting investor confidence [Source: Edgar SEC Filings - Jan 26, 2026](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000044/crwv-20260123.htm).

### Product/Tech Launches
- **Date:** January 26, 2026
- **Facts:** CoreWeave announced an expansion of its collaboration with NVIDIA to accelerate the buildout of more than 5 gigawatts of AI factories by 2030. This includes adopting NVIDIA's CPU and storage platforms.
- **Investment Implications:** Bullish. The collaboration with NVIDIA is expected to enhance CoreWeave's technological capabilities and market position in AI infrastructure [Source: Benzinga - Jan 26, 2026](https://www.benzinga.com/node/50123767?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

### M&A and Partnerships
- **Date:** January 26, 2026
- **Facts:** NVIDIA invested $2 billion in CoreWeave's Class A common stock at $87.20 per share, reflecting confidence in CoreWeave's growth strategy.
- **Investment Implications:** Bullish. NVIDIA's investment is a strong endorsement of CoreWeave's business model and growth potential, likely to attract further investor interest [Source: The Fly - Jan 26, 2026](https://app.bigdata.com/files#?document=8A18A14BF303B27A908A3D7C53C9847D).

### Regulatory/Legal Updates
- **Date:** January 29, 2026
- **Facts:** Multiple class action lawsuits have been filed against CoreWeave, alleging securities fraud related to misrepresentations about its infrastructure capabilities and delays in data center completion.
- **Investment Implications:** Bearish. Legal challenges could pose financial and reputational risks, potentially impacting stock performance [Source: Benzinga - Jan 29, 2026](https://www.benzinga.com/node/50231987?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

### Management Changes
- **No significant management changes reported in the last 30 days.**

### Other Material Events
- **Date:** January 27, 2026
- **Facts:** Deutsche Bank upgraded CoreWeave to Buy from Hold, raising the price target to $140, citing a solid medium-term outlook.
- **Investment Implications:** Bullish. The upgrade reflects positive analyst sentiment and could drive stock price appreciation [Source: Benzinga - Jan 27, 2026](https://www.benzinga.com/node/50151658?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

### Sources
- [Edgar SEC Filings - Jan 26, 2026](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000044/crwv-20260123.htm)
- [Benzinga - Jan 26, 2026](https://www.benzinga.com/node/50123767?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)
- [The Fly - Jan 26, 2026](https://app.bigdata.com/files#?document=8A18A14BF303B27A908A3D7C53C9847D)
- [Benzinga - Jan 29, 2026](https://www.benzinga.com/node/50231987?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)
- [Benzinga - Jan 27, 2026](https://www.benzinga.com/node/50151658?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)

### Example 5: Multi-Source Query

Combine local and external data:

In [15]:
# Example 4: Multi-source comprehensive analysis
query = """For our NVIDIA holdings:
1. Check our internal database to see which portfolios hold NVDA and how much
2. Search our internal research for our investment thesis
3. Use Bigdata tools to find recent news about NVIDIA
4. Provide a comprehensive summary combining all sources"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
#display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

## Example 6 : Company Briefing 

In [16]:
query = """
Summarize recent developments for CoreWeave (last 30 days).
**Steps:**
1. Call find_companies and get the company id
2. Call bigdata_tearsheet and get business context
3. Use bigdata_search and find news in the last 30 days
4. Categorize findings
**Categories:**
- Financial results
- Product/tech launches
- M&A and partnerships
- Regulatory/legal updates
- Management changes
- Other material events
For each: Date, facts, investment implications (bullish/bearish/neutral).
Please add inline source attribution.
"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)



In [17]:
# Display response
display_response(result)

# Display citations from Bigdata.com sources
display_citations(result)

Here's a summary of recent developments for CoreWeave over the last 30 days, categorized by type of event:

### Financial Results
- **Date:** January 29, 2026
- **Facts:** CoreWeave's stock price has seen significant volatility, with a 30.90% increase over the past month. The company reported a Q3 2025 EPS of -$0.08, beating estimates of -$0.40, and revenue of $1.36 billion, surpassing the expected $1.29 billion.
- **Investment Implications:** Bullish. The positive earnings surprise and revenue beat suggest stronger-than-expected financial performance, potentially boosting investor confidence [Source: Edgar SEC Filings - Jan 26, 2026](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000044/crwv-20260123.htm).

### Product/Tech Launches
- **Date:** January 26, 2026
- **Facts:** CoreWeave announced an expansion of its collaboration with NVIDIA to accelerate the buildout of more than 5 gigawatts of AI factories by 2030. This includes adopting NVIDIA's CPU and storage platforms.
- **Investment Implications:** Bullish. The collaboration with NVIDIA is expected to enhance CoreWeave's technological capabilities and market position in AI infrastructure [Source: Benzinga - Jan 26, 2026](https://www.benzinga.com/node/50123767?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

### M&A and Partnerships
- **Date:** January 26, 2026
- **Facts:** NVIDIA invested $2 billion in CoreWeave's Class A common stock at $87.20 per share, reflecting confidence in CoreWeave's growth strategy.
- **Investment Implications:** Bullish. NVIDIA's investment is a strong endorsement of CoreWeave's business model and growth potential, likely to attract further investor interest [Source: The Fly - Jan 26, 2026](https://app.bigdata.com/files#?document=8A18A14BF303B27A908A3D7C53C9847D).

### Regulatory/Legal Updates
- **Date:** January 29, 2026
- **Facts:** Multiple class action lawsuits have been filed against CoreWeave, alleging securities fraud related to misrepresentations about its infrastructure capabilities and delays in data center completion.
- **Investment Implications:** Bearish. Legal challenges could pose financial and reputational risks, potentially impacting stock performance [Source: Benzinga - Jan 29, 2026](https://www.benzinga.com/node/50231987?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

### Management Changes
- **No significant management changes reported in the last 30 days.**

### Other Material Events
- **Date:** January 27, 2026
- **Facts:** Deutsche Bank upgraded CoreWeave to Buy from Hold, raising the price target to $140, citing a solid medium-term outlook.
- **Investment Implications:** Bullish. The upgrade reflects positive analyst sentiment and could drive stock price appreciation [Source: Benzinga - Jan 27, 2026](https://www.benzinga.com/node/50151658?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

### Sources
- [Edgar SEC Filings - Jan 26, 2026](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000044/crwv-20260123.htm)
- [Benzinga - Jan 26, 2026](https://www.benzinga.com/node/50123767?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)
- [The Fly - Jan 26, 2026](https://app.bigdata.com/files#?document=8A18A14BF303B27A908A3D7C53C9847D)
- [Benzinga - Jan 29, 2026](https://www.benzinga.com/node/50231987?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)
- [Benzinga - Jan 27, 2026](https://www.benzinga.com/node/50151658?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)

## 9️⃣ Understanding the Agent Flow

The agent follows this process:

1. **Parse Query** → Understand what information is needed
2. **Plan** → Decide which tools to use
3. **Execute** → Call selected tools in sequence or parallel
4. **Synthesize** → Combine results into coherent answer
5. **Iterate** → If more information needed, repeat steps 2-4

**Tool Selection Logic:**
- Portfolio/holdings questions → `internal_query_database` or `internal_portfolio_summary`
- Internal research → `internal_search_research`
- External market data → Bigdata MCP tools (automatically discovered)

**Trace Visibility:**
- All tool calls are logged to LangSmith for debugging
- View traces at: https://smith.langchain.com

## 🔟 Key Benefits of This Architecture

### 1. **Automatic Tool Discovery**
- MCP server exposes tools dynamically
- No code changes needed when Bigdata.com adds new tools
- Agent automatically learns about new capabilities

### 2. **Unified Interface**
- Single agent interface for all data sources
- Consistent tool calling pattern
- Easy to add more MCP servers or local tools

### 3. **Stateful Reasoning**
- LangGraph maintains conversation state
- Agent can reference previous tool results
- Multi-turn reasoning supported

### 4. **Observability**
- LangSmith tracing shows full execution flow
- Easy to debug tool selection and results
- Performance monitoring built-in

## 🎯 Next Steps

**Extend this architecture:**

1. **Add More MCP Servers**
   ```python
   mcp_client = MultiServerMCPClient({
       "bigdata": {...},
       "other_mcp_server": {...}
   })
   ```

2. **Custom Local Tools**
   - Create @tool decorated functions
   - Add to tool list before agent creation

3. **Advanced Graph Patterns**
   - Use `StateGraph` for custom control flow
   - Add conditional edges for routing logic
   - Implement human-in-the-loop

4. **Persistent Memory**
   - Add checkpointer for conversation history
   - Use `MemorySaver` or Redis for state persistence

**References:**
- LangGraph: https://langchain-ai.github.io/langgraph/
- MCP Adapters: https://reference.langchain.com/python/langchain_mcp_adapters/
- Bigdata MCP: https://docs.bigdata.com/mcp-reference/

---

## 📚 Additional Resources

- **Bigdata.com API Documentation**: https://docs.bigdata.com
- **LangGraph Documentation**: https://langchain-ai.github.io/langgraph/
- **MCP Protocol Spec**: https://modelcontextprotocol.io
- **LangSmith Tracing**: https://smith.langchain.com

**Questions?** Contact: support@bigdata.com